In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import sys

project_root = '/content/drive/MyDrive/GNNs/HIV inhibitors-GNN'
os.makedirs(project_root, exist_ok=True)
os.chdir(project_root)
sys.path.append(project_root)

In [3]:
!pip install torch==2.4.0

import torch
torch_version = torch.__version__
print("PyTorch version:", torch_version)

!pip install rdkit deepchem

!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv \
  -f https://data.pyg.org/whl/torch-${torch_version}.html

!pip install torch_geometric


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:

# !pip install -q mlflow pyngrok

# import os
# from getpass import getpass
# import mlflow
# from pyngrok import ngrok
# import time
# import subprocess

# BASE_DIR = "/content/drive/MyDrive/GNNs/HIV inhibitors-GNN"

# os.makedirs(f"{BASE_DIR}/outputs/mlruns", exist_ok=True)
# os.makedirs(f"{BASE_DIR}/outputs/artifacts", exist_ok=True)

# MLFLOW_TRACKING_URI = f"sqlite:///{BASE_DIR}/outputs/mlruns/mlflow.db"
# mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# EXPERIMENT_NAME = "HIV_GNN_Classification"

# artifact_location = f"file://{BASE_DIR}/outputs/artifacts"

# try:
#     experiment_id = mlflow.create_experiment(
#         EXPERIMENT_NAME,
#         artifact_location=artifact_location
#     )
#     print("Created experiment:", experiment_id)
# except:
#     experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
#     experiment_id = experiment.experiment_id

# mlflow.set_experiment(EXPERIMENT_NAME)

# print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
# print(f"Experiment: {EXPERIMENT_NAME}")

# print("\nngrok authtoken............")
# ngrok_token = getpass("Paste your ngrok authtoken here and press Enter: ")

# if ngrok_token.strip():
#     !ngrok config add-authtoken {ngrok_token}
#     print("ngrok authtoken saved!")
# else:
#     print("No token entered. Public UI will not work.")

# # === Kill old processes (clean start) ===
# ngrok.kill()
# !pkill -f "mlflow ui" || true
# time.sleep(2)

# print("\nStarting MLflow UI...")

# # ✅ FIXED + IMPROVED command
# subprocess.Popen([
#     "mlflow", "ui",
#     "--host", "0.0.0.0",
#     "--port", "5000",
#     "--workers", "1",
#     "--allowed-hosts", "*",
#     "--backend-store-uri", MLFLOW_TRACKING_URI,
#     "--default-artifact-root", f"file://{BASE_DIR}/outputs/artifacts"
# ])

# # Wait until the port is actually listening
# print("Waiting for MLflow to bind to port 5000...")
# for i in range(15):
#     time.sleep(1)
#     import socket
#     sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
#     result = sock.connect_ex(('127.0.0.1', 5000))
#     sock.close()
#     if result == 0:
#         print(f"✅ MLflow UI is ready on port 5000 (took {i+1}s)")
#         break
#     if i == 14:
#         print("⚠️ MLflow did not start in time. Check the next lines.")

# time.sleep(2)

# # Start ngrok
# try:
#     public_url = ngrok.connect(5000, "http").public_url
#     print(f"\n🎉 MLflow UI is live at: {public_url}")
#     print("   → Open this link in a new tab")
#     print("   → Keep this Colab cell running (do not close the tab)")
# except Exception as e:
#     print(f"\n❌ Could not create ngrok tunnel: {e}")

# # === Diagnostic (very useful) ===
# print("\n=== Running processes ===")
# !ps aux | grep mlflow
# print("\n=== Port 5000 status ===")
# !lsof -i :5000


# Install MLflow and Ngrok
!pip install -q mlflow pyngrok

import os
import mlflow
from pyngrok import ngrok
import subprocess
import socket
import time
from getpass import getpass

# ---------------------------
# Directories
# ---------------------------
BASE_DIR = "/content/drive/MyDrive/GNNs/HIV inhibitors-GNN"
MLRUNS_DIR = f"{BASE_DIR}/outputs/mlruns"
ARTIFACT_DIR = f"{BASE_DIR}/outputs/artifacts"

os.makedirs(MLRUNS_DIR, exist_ok=True)
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# ---------------------------
# MLflow tracking URI
# ---------------------------
MLFLOW_TRACKING_URI = f"sqlite:///{MLRUNS_DIR}/mlflow.db"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print("Tracking URI:", mlflow.get_tracking_uri())

# ---------------------------
# Experiment setup
# ---------------------------
EXPERIMENT_NAME = "HIV_GNN_Classification"
artifact_location = f"file://{ARTIFACT_DIR}"

# Try to get existing experiment first
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME, artifact_location=artifact_location)
    print(f"Created experiment: {EXPERIMENT_NAME} (ID: {experiment_id})")
else:
    experiment_id = experiment.experiment_id
    print(f"Using existing experiment: {EXPERIMENT_NAME} (ID: {experiment_id})")

mlflow.set_experiment(EXPERIMENT_NAME)

# ---------------------------
# Ngrok setup
# ---------------------------
token = getpass("Paste your ngrok authtoken: ")
if token.strip():
    !ngrok config add-authtoken {token}

# Clean old processes
ngrok.kill()
!pkill -f mlflow || true
time.sleep(2)

# ---------------------------
# Start MLflow server
# ---------------------------
print("Starting MLflow server...")
subprocess.Popen([
    "mlflow", "server",
    "--host", "0.0.0.0",
    "--port", "5000",
    "--backend-store-uri", MLFLOW_TRACKING_URI,
    "--default-artifact-root", f"file://{ARTIFACT_DIR}",
    "--workers", "1",
    "--allowed-hosts", "*",
    "--cors-allowed-origins", "*"
])

# Wait for server
for i in range(20):
    time.sleep(1)
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    if sock.connect_ex(("127.0.0.1", 5000)) == 0:
        print("✅ MLflow server ready")
        break
    sock.close()

# ---------------------------
# Start ngrok tunnel
# ---------------------------
public_url = ngrok.connect(5000).public_url
print("\n🎉 MLflow UI:")
print(public_url)
print("\nKeep this Colab cell running!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 822.0/822.0 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 10.8 MB/s eta 0:00:00
Tracking URI: sqlite:////content/drive/MyDrive/GNNs/HIV inhibitors-GNN/outputs/mlruns/mlflow.db
Using existing experiment: HI

In [5]:
%run src/training/train.py --epochs 100 --use_pos_weight 0 --use_weighted_sampler 1

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.
Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead
wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.


Using pos_weight: False, Using weighted sampler: True
loading datasets.........

Derived parameters......

Get loaders..................
train_set_size:32568 | val_set_size:4440
Class Counts: [31421  1147]
Class distribution: 31421 negatives(0), 1147 positives(1)
pos_weight = 1.0

Create Model.................

Create loss function and Optimizer.................

Load Checkpoint.................
no checkpoint found. starting from epoch 0.
Starting new MLflow run: c32199b406844dfa85ea3cf941df96f2

Log Params in MLflow.........................



Start training loop.....................




Training......:   0%|          | 0/509 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
Training......: 100%|██████████| 509/509 [00:14<00:00, 34.29it/s]


Type:train| Epoch:0 | AP: 0.6608 | AUC: 0.6457 | Recall@0.1: 0.998147233201581 | Precision@0.1: 0.49713934174100277 | Recall@0.2: 0.9930212450592886 | Precision@0.2: 0.49852726878119863 | Recall@0.3: 0.9563364624505929 | Precision@0.3: 0.5076383425124574 | Recall@0.5: 0.5226037549407114 | Precision@0.5: 0.6257024548950014 

Epoch: 0 | Train loss: 0.6616715524273346




Training......:   0%|          | 0/509 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
Training......: 100%|██████████| 509/509 [00:14<00:00, 35.48it/s]


...

Validation-loss model......: 100%|██████████| 70/70 [00:01<00:00, 48.70it/s]


Final Results =>  Type:Validation-loss model| AP: 0.1889 | AUC: 0.7376 | Final Recall@0.1: 0.8881578947368421 | Final Precision@0.1: 0.042884371029224905 | Final Recall@0.2: 0.6381578947368421 | Final Precision@0.2: 0.07376425855513308 | Final Recall@0.3: 0.4934210526315789 | Final Precision@0.3: 0.1273344651952462 | Final Recall@0.5: 0.21052631578947367 | Final Precision@0.5: 0.2857142857142857 



Test-loss model......: 100%|██████████| 65/65 [00:00<00:00, 66.87it/s]


Final Results =>  Type:Test-loss model| AP: 0.2067 | AUC: 0.7427 | Final Recall@0.1: 0.875 | Final Precision@0.1: 0.04452296819787986 | Final Recall@0.2: 0.6319444444444444 | Final Precision@0.2: 0.07933740191804708 | Final Recall@0.3: 0.4583333333333333 | Final Precision@0.3: 0.13692946058091288 | Final Recall@0.5: 0.20833333333333334 | Final Precision@0.5: 0.3191489361702128 



<Figure size 640x480 with 0 Axes>